# 03 · E2: LLM local con instrucciones (zero-shot, few-shot fijo y few-shot dinámico)

### Qué es este experimento

En E1 **entrenamos** un modelo con miles de oraciones anotadas. Aquí hacemos lo contrario: tomamos un **LLM** (un modelo generativo, como ChatGPT, pero corriendo en el propio Mac con Ollama) y **no lo entrenamos**. Solo le damos instrucciones y, a veces, algunos ejemplos resueltos. Esta es la parte de **IA generativa** del proyecto.

**Analogía:** E1 es un especialista que hizo una beca de un año marcando fichas. E2 es un médico general muy culto al que le pasamos una hoja con instrucciones y le pedimos que marque las fichas en el momento.

### Las tres estrategias

| Estrategia | Qué recibe el LLM | Analogía |
|---|---|---|
| **zero-shot** | Solo la guía de anotación | Instrucciones escritas, sin ejemplos |
| **few-shot fijo** | La guía + los mismos 5 ejemplos resueltos para todas las oraciones | Instrucciones + 5 fichas modelo |
| **few-shot dinámico** | La guía + los 5 ejemplos de *train* **más parecidos** a cada oración | Instrucciones + las 5 fichas del archivo más parecidas a la que tiene en la mano |

La literatura dice que un LLM solo con instrucciones queda 3 a 15 puntos de F1 bajo un modelo ajustado, y que los ejemplos parecidos (*few-shot* dinámico) recortan parte de esa brecha. Queremos ver si eso se cumple en texto de APS chileno.

### Por qué un LLM local

Un CESFAM no puede mandar fichas de pacientes a un servidor externo sin resguardos (Ley 21.719). Un modelo de 7–8 mil millones de parámetros corre en un computador y los datos no salen de ahí. Usamos `llama3.1:8b` (Meta) y `qwen2.5:7b` (Alibaba).

### Conexión con el curso

- **Unidad 5–6:** modelos generativos y LLM; *prompting*, *in-context learning*, salida estructurada.
- **Unidad 7:** modelos fundacionales usados sin ajuste; recuperación de ejemplos (idea de RAG); alucinaciones.
- **Unidad 4:** evaluación con las mismas métricas que E1 (F1 estricto/laxo, *bootstrap*).

In [ ]:
from pathlib import Path
import sys, json, time, pickle
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ / "src"))

import yaml, random
import pandas as pd
import matplotlib.pyplot as plt
from datos.cwlc import cargar_split
from datos.auditoria import normalizar
from evaluacion import metricas
from modelos.e1_encoder import evaluar
from modelos import e2_llm as E2
from reproducibilidad import huella_completa, verificar_huellas_datos, agregar_semillas, sha256_bytes
from sistema import mantener_despierto, dejar_dormir

cfg_global = yaml.safe_load(open(RAIZ / "configs" / "experimentos.yaml"))
carpeta = RAIZ / "data" / "raw" / "cwlc"
train, dev, test = (cargar_split(carpeta, s) for s in ["train", "dev", "test"])
ok, difs = verificar_huellas_datos(carpeta, "cwlc")
assert ok, f"¡Los datos cambiaron! {difs}"
print(f"train={len(train)}  dev={len(dev)}  test={len(test)} · huellas verificadas")

## 1. La guía de anotación (el *prompt*)

El LLM no conoce las reglas con que los médicos anotaron el corpus. Por ejemplo: que en este corpus el signo "+" cuenta como abreviatura, o que un fármaco se marca **sin** la dosis. La literatura muestra que un *prompt* con guía de anotación mejora entre 5 y 37 puntos respecto a una instrucción simple.

La guía se redactó mirando **solo ejemplos de train** (nunca dev ni test). Le pedimos responder en **JSON** con una lista de menciones copiadas tal cual del texto. Ollama obliga al modelo a respetar ese formato (*salida estructurada*), y usamos **temperatura 0** (sin azar) y **semilla fija**, para que la misma pregunta dé siempre la misma respuesta.

In [ ]:
print(E2.GUIA)
print("\nHuella de la guía:", sha256_bytes(E2.GUIA.encode())[:16])

## 2. ¿Están los modelos?

Igual que con los modelos de Hugging Face, registramos el **digest** exacto de cada modelo de Ollama: si mañana se actualiza `qwen2.5:7b`, sabremos que no es el mismo.

In [ ]:
import ollama
MODELOS = cfg_global["llm_locales"]          # ["llama3.1:8b", "qwen2.5:7b"]
instalados = {m.get("model") or m.get("name") for m in ollama.list()["models"]}
for m in MODELOS:
    print(("✔ " if m in instalados else "✘ FALTA (ollama pull " + m + ") ") + m, E2.huella_ollama(m).get("digest", "")[:19])

## 3. Cómo se ve lo que recibe el LLM

Un ejemplo con *few-shot* dinámico, sobre una oración de **dev** (no del test). Los 5 ejemplos más parecidos se buscan en train con **BM25**, el mismo algoritmo que usan los buscadores para ordenar documentos por palabras en común.

In [ ]:
dinamico = E2.SelectorDinamico(train, k=cfg_global["k_ejemplos"])
o = next(x for x in dev if len(x.tokens) > 10 and x.entidades)
for m in E2.mensajes(o, dinamico(o))[1:]:
    print(f"[{m['role']}] {m['content'][:160]}")

## 4. Dos controles antes de gastar horas

**a) ¿Perdemos algo al leer la respuesta?** Si el LLM respondiera *perfecto* (le pasamos las respuestas correctas), ¿cuánto F1 sobrevive a la lectura del JSON y a la ubicación de cada mención en la oración? Debería ser ~1. Lo que falte es error nuestro, no del LLM.

**b) Fuga en el *few-shot* dinámico.** En la auditoría (notebook 01b) vimos que ~10% de las oraciones del test están repetidas en train. Con *few-shot* dinámico, para esas oraciones el ejemplo más parecido es **la misma oración con su respuesta**: el LLM solo tiene que copiar. Por eso, para E2, el F1 **sin duplicados** es especialmente importante.

In [ ]:
resp_perfectas = {o.id: {"respuesta": E2.respuesta_gold(o), "segundos": 0} for o in test}
ev = E2.evaluar_respuestas(resp_perfectas, test)
for t in E2.TIPOS_E2:
    g = [e for o in test for e in o.entidades if e[3] == t]
    print(f"a) {t:<13} F1 con respuestas perfectas = {metricas.contar(g, ev['pred'][t], 'estricto').f1:.4f}")
n = sum(normalizar(dinamico(o)[-1].texto) == normalizar(o.texto) for o in test)
print(f"b) Oraciones del test cuyo ejemplo más parecido es idéntico: {n} ({100 * n / len(test):.1f}%)")

## 5. Configuración

- **`MODO = "rapido"`**: 100 oraciones del test elegidas al azar (semilla 42), para medir tiempos y revisar que todo funcione (~30–40 min).
- **`MODO = "completo"`**: las 990 oraciones del test (varias horas; el tiempo real se estima en el modo rápido).

**Semillas.** Con temperatura 0 el LLM no tiene azar: zero-shot y dinámico se corren **una vez**. El azar está en **qué 5 ejemplos** se eligen para el *few-shot* fijo: por eso esa estrategia se repite con 3 semillas (3 sets de ejemplos distintos) y se reporta media ± DE. En la sección 7 se comprueba que repetir una pregunta da la misma respuesta.

In [ ]:
MODO = "rapido"          # "rapido" para probar | "completo" para reportar

K = cfg_global["k_ejemplos"]
SEMILLAS_FIJO = cfg_global["semillas"]                     # [42, 43, 44]
if MODO == "rapido":
    oraciones = sorted(random.Random(42).sample(test, 100), key=lambda o: o.id)
else:
    oraciones = test
CORRIDAS = [("zero_shot", None), ("few_shot_dinamico", None)] + [("few_shot_fijo", s) for s in SEMILLAS_FIJO]
SALIDA = RAIZ / "results" / f"e2_{MODO}"
CACHE = RAIZ / "results" / "cache" / f"e2_{MODO}"           # respuestas crudas (no van a git)
print(f"{MODO}: {len(oraciones)} oraciones × {len(CORRIDAS)} corridas × {len(MODELOS)} modelos = "
      f"{len(oraciones) * len(CORRIDAS) * len(MODELOS)} llamadas al LLM")

## 6. Correr

Cada respuesta se guarda apenas llega (en `results/cache/`), así que si se corta, al volver a correr la celda sigue donde quedó. La celda mantiene el Mac despierto. Al terminar cada corrida se evalúa con **las mismas funciones que E1**, incluidos el F1 sin duplicados y el *recall* en menciones vistas vs nuevas, y se guarda un JSON con su huella.

In [ ]:
despierto = mantener_despierto()
textos_test = {o.id: o for o in oraciones}
for modelo in MODELOS:
    for estrategia, semilla in CORRIDAS:
        nombre = f"{modelo.replace(':', '-')}_{estrategia}" + (f"_s{semilla}" if semilla else "")
        ruta = SALIDA / f"{nombre}.json"
        if ruta.exists():
            print(f"(ya existe) {ruta.name}"); continue
        print(f"\n=== {modelo} · {estrategia}" + (f" · semilla {semilla}" if semilla else "") + " ===")
        if estrategia == "zero_shot":
            selector = None
        elif estrategia == "few_shot_dinamico":
            selector = dinamico
        else:
            selector = E2.SelectorFijo(train, K, semilla)
        cache = E2.Cache(CACHE / f"{nombre}.jsonl")
        t0 = time.time()
        resp = E2.correr(modelo, oraciones, selector, cache, semilla=semilla or 42)
        ev = E2.evaluar_respuestas(resp, oraciones)
        res = {"modelo": modelo, "estrategia": estrategia, "semilla": semilla,
               "tasa_alucinacion": ev["tasa_alucinacion"], "json_invalidos": ev["json_invalidos"],
               "segundos_por_oracion": ev["segundos_por_oracion"], "por_tipo": {}}
        for tipo in E2.TIPOS_E2:
            gold = [e for o in oraciones for e in o.entidades if e[3] == tipo]
            res["por_tipo"][tipo] = evaluar(gold, ev["pred"][tipo], oraciones, tipo, train=train)
        config = {"modelo": modelo, "estrategia": estrategia, "semilla": semilla, "k": K, "temperatura": 0,
                  "num_ctx": 8192, "guia": sha256_bytes(E2.GUIA.encode()), "modo": MODO,
                  "oraciones": sha256_bytes(json.dumps([o.id for o in oraciones]).encode()),
                  "ejemplos_fijos": [o.id for o in selector.ejemplos] if estrategia == "few_shot_fijo" else None}
        huella = huella_completa(config, carpeta, splits={"train": train, "test": test})
        huella["ollama"] = E2.huella_ollama(modelo)
        SALIDA.mkdir(parents=True, exist_ok=True)
        ruta.write_text(json.dumps({"config": config, "resultados": res, "huella": huella}, indent=2, ensure_ascii=False, default=str))
        f1s = "  ".join(f"{t[:4]}={res['por_tipo'][t]['estricto']['f1']:.3f}" for t in E2.TIPOS_E2)
        print(f"  F1 estricto: {f1s} · alucinación {ev['tasa_alucinacion']:.1%} · {(time.time() - t0) / 60:.1f} min")
dejar_dormir(despierto)

## 7. ¿Es reproducible?

Volvemos a preguntar 20 oraciones (sin usar la caché) y comparamos con las respuestas guardadas. Con temperatura 0 y semilla fija deberían ser idénticas, letra por letra.

In [ ]:
m = MODELOS[0]
cache = E2.Cache(CACHE / f"{m.replace(':', '-')}_zero_shot.jsonl")
iguales = sum(E2.llamar_ollama(m, E2.mensajes(o, []), 42) == cache.datos[o.id]["respuesta"] for o in oraciones[:20])
print(f"{m}, zero-shot: {iguales}/20 respuestas idénticas al repetir la pregunta")

## 8. Resultados

Cómo leer las columnas (además de las de E1):

- **Alucinación:** proporción de menciones que el LLM inventó (no aparecen en la oración). Un modelo ajustado como E1 nunca alucina; un LLM sí puede.
- **JSON inválidos:** respuestas que no se pudieron leer.
- **s/oración:** tiempo por oración en el Mac.

In [ ]:
filas = []
for f in sorted(SALIDA.glob("*.json")):
    d = json.loads(f.read_text()); r = d["resultados"]
    for tipo, m in r["por_tipo"].items():
        filas.append({"modelo": r["modelo"], "estrategia": r["estrategia"], "semilla": r["semilla"], "tipo": tipo,
                      "F1 estricto": m["estricto"]["f1"], "F1 laxo": m["laxo"]["f1"], "P": m["estricto"]["precision"],
                      "R": m["estricto"]["recall"], "F1 sin dup.": m["test_sin_duplicados"]["f1_estricto"],
                      "recall vistas": m["menciones_vistas"]["recall_estricto"], "recall nuevas": m["menciones_nuevas"]["recall_estricto"],
                      "alucinación": r["tasa_alucinacion"], "JSON inválidos": r["json_invalidos"], "s/oración": r["segundos_por_oracion"]})
e2 = pd.DataFrame(filas)
METRICAS = ["F1 estricto", "F1 laxo", "F1 sin dup.", "recall vistas", "recall nuevas", "alucinación", "s/oración"]
def media_de(x):
    a = agregar_semillas(list(x)); return f"{a['media']:.3f} ± {a['de']:.3f}" if a["n"] > 1 else f"{a['media']:.3f}"
tabla_e2 = e2.groupby(["tipo", "modelo", "estrategia"])[METRICAS].agg(media_de)
tabla_e2

## 9. E2 frente a E1, sobre las mismas oraciones

Para comparar en igualdad, recalculamos el F1 de E1 **sobre exactamente las mismas oraciones** que vio el LLM, usando las predicciones guardadas del notebook 02 (media de sus 3 semillas).

In [ ]:
ids = {o.id for o in oraciones}
filas = []
for clave in ["espanol_clinico", "chileno_clinico"]:
    for tipo in E2.TIPOS_E2:
        gold = [e for o in oraciones for e in o.entidades if e[3] == tipo]
        f1s = []
        for s in cfg_global["semillas"]:
            p = RAIZ / "results" / "cache" / "e1_completo" / f"{clave}_{tipo}_s{s}.pkl"
            if p.exists():
                pred = [e for e in pickle.loads(p.read_bytes()) if e[0] in ids]
                f1s.append(metricas.contar(gold, pred, "estricto").f1)
        if f1s:
            filas.append({"tipo": tipo, "sistema": f"E1 {clave.split('_')[0]}", "F1 estricto": sum(f1s) / len(f1s)})
for (tipo, modelo, estrategia), g in e2.groupby(["tipo", "modelo", "estrategia"]):
    filas.append({"tipo": tipo, "sistema": f"E2 {modelo} · {estrategia}", "F1 estricto": g["F1 estricto"].mean()})
comparacion = pd.DataFrame(filas).pivot(index="sistema", columns="tipo", values="F1 estricto").round(3)
display(comparacion)

fig, ax = plt.subplots(figsize=(9, 4.5))
comparacion.T.plot.bar(ax=ax, rot=0, width=0.85)
ax.set_ylabel("F1 estricto (mismas oraciones)"); ax.set_ylim(0, 1)
ax.set_title("E1 (encoder ajustado) vs E2 (LLM local con instrucciones)")
ax.legend(fontsize=8, loc="upper left", bbox_to_anchor=(1, 1)); plt.tight_layout(); plt.show()

## 10. Conclusiones (completar después de la corrida)

- ¿Cuántos puntos de F1 quedan los LLM bajo E1? ¿Coincide con los 3–15 que reporta la literatura?
- ¿Cuánto mejora pasar de zero-shot a few-shot fijo y a dinámico?
- ¿Cuánto cae el dinámico **sin duplicados**? Si cae mucho, su ventaja venía de copiar oraciones repetidas.
- *Recall* en menciones **nuevas**: ¿el LLM generaliza mejor que E1 donde E1 solo memorizaba?
- Tasa de alucinación por modelo y estrategia.
- Diferencia entre F1 laxo y estricto: ¿el LLM encuentra las entidades pero corta mal los límites?
- Tiempo por oración: ¿es viable en un CESFAM?

**Siguiente:** E4, el LLM como anotador para entrenar un modelo pequeño.